In [1]:
import pandas as pd

In [28]:
import nltk
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
import matplotlib.pyplot as plt

In [35]:
from nltk import FreqDist

In [39]:
from nltk.tokenize import sent_tokenize

In [2]:
file_path = 'apartments_for_rent_classified_10K.csv'
df = pd.read_csv('apartments_for_rent_classified_10K.csv', sep=';', encoding='cp1252', on_bad_lines='skip')

In [3]:
df.head()

,id,category,title,body,amenities,bathrooms,bedrooms,currency,fee,has_photo,...,price_display,price_type,square_feet,address,cityname,state,latitude,longitude,source,time
0,5668626895,housing/rent/apartment,"Studio apartment 2nd St NE, Uhland Terrace NE,...","This unit is located at second St NE, Uhland T...",NaN,NaN,0.0,USD,No,Thumbnail,...,$790,Monthly,101,NaN,Washington,DC,38.9057,-76.9861,RentLingo,1577359415
1,5664597177,housing/rent/apartment,Studio apartment 814 Schutte Road,"This unit is located at 814 Schutte Road, Evan...",NaN,NaN,1.0,USD,No,Thumbnail,...,$425,Monthly,106,814 Schutte Rd,Evansville,IN,37.9680,-87.6621,RentLingo,1577017063
2,5668626833,housing/rent/apartment,"Studio apartment N Scott St, 14th St N, Arling...","This unit is located at N Scott St, 14th St N,...",NaN,1.0,0.0,USD,No,Thumbnail,...,"$1,390",Monthly,107,NaN,Arlington,VA,38.8910,-77.0816,RentLingo,1577359410
3,5659918074,housing/rent/apartment,Studio apartment 1717 12th Ave,"This unit is located at 1717 12th Ave, Seattle...",NaN,1.0,0.0,USD,No,Thumbnail,...,$925,Monthly,116,1717 12th Avenue,Seattle,WA,47.6160,-122.3275,RentLingo,1576667743
4,5668626759,housing/rent/apartment,"Studio apartment Washington Blvd, N Cleveland ...","This unit is located at Washington Blvd, N Cle...",NaN,NaN,0.0,USD,No,Thumbnail,...,$880,Monthly,125,NaN,Arlington,VA,38.8738,-77.1055,RentLingo,1577359401


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 22 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             10000 non-null  int64  
 1   category       10000 non-null  object 
 2   title          10000 non-null  object 
 3   body           10000 non-null  object 
 4   amenities      6451 non-null   object 
 5   bathrooms      9966 non-null   float64
 6   bedrooms       9993 non-null   float64
 7   currency       10000 non-null  object 
 8   fee            10000 non-null  object 
 9   has_photo      10000 non-null  object 
 10  pets_allowed   5837 non-null   object 
 11  price          10000 non-null  int64  
 12  price_display  10000 non-null  object 
 13  price_type     10000 non-null  object 
 14  square_feet    10000 non-null  int64  
 15  address        6673 non-null   object 
 16  cityname       9923 non-null   object 
 17  state          9923 non-null   object 
 18  latitud

In [5]:
    duplicate_rows = df.duplicated().sum()
    print(duplicate_rows)

0


In [6]:
df.describe()

,id,bathrooms,bedrooms,price,square_feet,latitude,longitude,time
count,1.000000e+04,9966.000000,9993.000000,10000.000000,10000.000000,9990.000000,9990.000000,1.000000e+04
mean,5.623396e+09,1.380544,1.744021,1486.277500,945.810500,37.695162,-94.652247,1.574891e+09
std,7.021025e+07,0.615410,0.942354,1076.507968,655.755736,5.495851,15.759805,3.762395e+06
min,5.508654e+09,1.000000,0.000000,200.000000,101.000000,21.315500,-158.022100,1.568744e+09
25%,5.509248e+09,1.000000,1.000000,949.000000,649.000000,33.679850,-101.301700,1.568781e+09
50%,5.668610e+09,1.000000,2.000000,1270.000000,802.000000,38.809800,-93.651600,1.577358e+09
75%,5.668626e+09,2.000000,2.000000,1695.000000,1100.000000,41.349800,-82.209975,1.577359e+09
max,5.668663e+09,8.500000,9.000000,52500.000000,40000.000000,61.594000,-70.191600,1.577362e+09


In [7]:
df['category'].unique()

array(['housing/rent/apartment', 'housing/rent/home',
       'housing/rent/short_term'], dtype=object)

In [8]:
df['title'].unique()

array(['Studio apartment 2nd St NE, Uhland Terrace NE, Washington, DC 20002',
       'Studio apartment 814 Schutte Road',
       'Studio apartment N Scott St, 14th St N, Arlington, VA 22209', ...,
       'Six BR 9908 Bentcross Drive', 'One BR in New York NY 10069',
       'Beautiful Lawrenceville Apartment for rent'], dtype=object)

In [9]:
df['body'].unique()

array(['This unit is located at second St NE, Uhland Terrace NE, Washington, DC 20002, Washington, 20002, DCMonthly rental rates range from $790 - $1090We have studio units available for rent',
       'This unit is located at 814 Schutte Road, Evansville, 47712, INMonthly rental rates range from $425 - $445We have studio - 1 beds units available for rent',
       'This unit is located at N Scott St, 14th St N, Arlington, VA 22209, Arlington, 22209, VAMonthly rental rates range from $1390We have studio units available for rent',
       ...,
       'This unit is located at 9908 Bentcross Drive, Potomac, 20854, MDMonthly rental rates range from $11000We have 6 beds units available for rent',
       "Monthly Rent$4,605 -to $4,790AmenitiesThe Aldyn offers some of the finest amenities amongst Upper West Side apartments. The club-style way of apartment living offers residents countless amenities right at their doorstep - the cornerstone being the 40,000 sq. feet La Palestra Athletic Club and 

In [10]:
df['body'].unique().shape

(9961,)

In [11]:
df['amenities'].unique()

array([nan, 'Dishwasher,Elevator,Patio/Deck,Pool,Storage', 'Refrigerator',
       ...,
       'Cable or Satellite,Dishwasher,Fireplace,Parking,Patio/Deck,Refrigerator,Wood Floors',
       'Elevator,Gym,Parking,Patio/Deck,Pool,Storage,Tennis,View',
       'Basketball,Cable or Satellite,Doorman,Hot Tub,Internet Access,Parking,Playground,Pool,Storage,Washer Dryer'],
      dtype=object)

In [12]:
df['currency'].unique()

array(['USD'], dtype=object)

In [13]:
df['fee'].unique()

array(['No'], dtype=object)

In [14]:
df['fee'].unique()

array(['No'], dtype=object)

In [15]:
df['has_photo'].unique()

array(['Thumbnail', 'Yes', 'No'], dtype=object)

In [16]:
df['pets_allowed'].unique()

array([nan, 'Cats,Dogs', 'Cats', 'Dogs'], dtype=object)

In [17]:
df['price_display'].unique()

array(['$790', '$425', '$1,390', ..., '$19,500', '$25,000', '$4,790'],
      dtype=object)

In [18]:
df['price_type'].unique()

array(['Monthly', 'Weekly', 'Monthly|Weekly'], dtype=object)

In [19]:
df['address'].unique()

array([nan, '814 Schutte Rd', '1717 12th Avenue', ...,
       '5407 Abbott Place  Abbott', '256 Las Entradas',
       '9908 Bentcross Dr'], dtype=object)

In [20]:
df['cityname'].unique()

array(['Washington', 'Evansville', 'Arlington', ..., 'Saint Leonard',
       'Chaska', 'Bella Vista'], dtype=object)

In [21]:
df['state'].unique()

array(['DC', 'IN', 'VA', 'WA', 'NY', 'CA', 'AZ', 'NC', 'TX', 'GA', 'FL',
       nan, 'AL', 'MD', 'CO', 'NM', 'IL', 'TN', 'AK', 'MA', 'NJ', 'OR',
       'DE', 'PA', 'IA', 'SC', 'MN', 'MI', 'KY', 'WI', 'OH', 'CT', 'RI',
       'NV', 'UT', 'MO', 'OK', 'NH', 'NE', 'LA', 'ND', 'AR', 'KS', 'ID',
       'HI', 'MT', 'VT', 'SD', 'WV', 'MS', 'ME', 'WY'], dtype=object)

In [22]:
df['source'].unique()

array(['RentLingo', 'Listanza', 'ListedBuy', 'RentDigs.com', 'GoSection8',
       'RealRentals', 'RENTOCULAR', 'rentbits', 'Home Rentals',
       'Real Estate Agent', 'RENTCafé', 'tenantcloud'], dtype=object)

In [23]:
df['body'].iloc[1]

'This unit is located at 814 Schutte Road, Evansville, 47712, INMonthly rental rates range from $425 - $445We have studio - 1 beds units available for rent'

In [24]:
df.iloc[1]

id                                                      5664597177
category                                    housing/rent/apartment
title                            Studio apartment 814 Schutte Road
body             This unit is located at 814 Schutte Road, Evan...
amenities                                                      NaN
bathrooms                                                      NaN
bedrooms                                                       1.0
currency                                                       USD
fee                                                             No
has_photo                                                Thumbnail
pets_allowed                                                   NaN
price                                                          425
price_display                                                 $425
price_type                                                 Monthly
square_feet                                                   

In [38]:
with open("title_column_texts.txt", "r", encoding = "utf-8") as f:
    text = f.read()

In [41]:
text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
tokens = word_tokenize(text)

In [42]:
total_tokens = len(tokens)
unique_tokens = len(set(tokens))
print(f'Total tokens: {total_tokens}')
print(f'unique tokens: {unique_tokens}')

Total tokens: 57743
unique tokens: 9478


In [43]:
print(tokens[:10])

['Studio', 'apartment', '2nd', 'St', 'NE', 'Uhland', 'Terrace', 'NE', 'Washington', 'DC']


In [45]:
fdist = FreqDist(tokens)
print("\n--- Top 15 Most Common Words ---")
print(fdist.most_common(50))


--- Top 15 Most Common Words ---
[('BR', 7200), ('One', 3922), ('Two', 2154), ('Three', 990), ('Studio', 910), ('Apartment', 897), ('apartment', 892), ('St', 891), ('Street', 845), ('Ave', 817), ('in', 789), ('Drive', 567), ('BA', 567), ('N', 488), ('S', 481), ('Pet', 432), ('OK', 431), ('Avenue', 419), ('E', 418), ('W', 403), ('Four', 381), ('for', 366), ('Road', 358), ('Rd', 337), ('Dr', 332), ('rent', 278), ('Blvd', 268), ('to', 229), ('a', 224), ('of', 194), ('2', 179), ('Lane', 174), ('South', 173), ('the', 170), ('Apartments', 165), ('location', 164), ('The', 161), ('and', 159), ('Unit', 157), ('Way', 150), ('Garage', 150), ('1', 146), ('Car', 146), ('West', 142), ('Single', 131), ('East', 130), ('City', 130), ('Spacious', 129), ('North', 127), ('Park', 122)]


In [48]:
# Save the content of the 'body' column to a text file
with open("body_column_texts_RentLingo.txt", "w", encoding='utf-8') as f:
    for entry in df[df['source'] == 'RentLingo']['body'].dropna().unique():
        f.write(entry.strip() + "\n")

In [49]:
with open("body_column_texts_RentLingo.txt", "r", encoding = "utf-8") as f:
    text = f.read()

['This unit is located at second St NE, Uhland Terrace NE, Washington, DC 20002, Washington, 20002, DCMonthly rental rates range from $790 - $1090We have studio units available for rent\nThis unit is located at 814 Schutte Road, Evansville, 47712, INMonthly rental rates range from $425 - $445We have studio - 1 beds units available for rent\nThis unit is located at N Scott St, 14th St N, Arlington, VA 22209, Arlington, 22209, VAMonthly rental rates range from $1390We have studio units available for rent\nThis unit is located at 1717 12th Ave, Seattle, 98122, WAMonthly rental rates range from $925We have studio units available for rent\nThis unit is located at Washington Blvd, N Cleveland St, Arlington, Arlington, 22201, VAMonthly rental rates range from $880We have studio units available for rent\nThis unit is located at 2432 Penmar Ave, Venice, 90291, CAMonthly rental rates range from $1800We have studio units available for rent\nThis unit is located at Oak St NW, 16th St NW, Washingto